# AI辅助设计RNA编辑器连接肽优化复现演示
本Notebook使用已提交的完整候选级记录，重新执行正式候选筛选、排序、标准结果生成和结构路径核验。它是CPU快速演示；从RFdiffusion开始的GPU全流程请运行仓库根目录的 `run_full.ps1`。

In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile
import pandas as pd

root = Path.cwd()
if not (root / 'results').exists():
    root = root.parent
assert (root / 'predict.py').is_file(), root
demo_dir = Path(tempfile.mkdtemp(prefix='adar2_linker_demo_'))
demo_csv = demo_dir / 'results.csv'
subprocess.run([sys.executable, str(root / 'predict.py'), '--input', str(root / 'results' / 'all_candidate_records.csv'), '--output', str(demo_csv)], cwd=root, check=True)
demo = pd.read_csv(demo_csv)
demo.head(10)

In [ ]:
required = ['candidate_id','track','linker_sequence','final_score_median','linker_rmsd_median_A','vina_median_kcal_mol','rf_backbone_pdb','boltz_model_0_pdb','boltz_model_1_pdb','boltz_model_2_pdb']
assert set(required).issubset(demo.columns)
def package_file_exists(value):
    path = (root / value).resolve()
    if os.name == 'nt':
        path = Path('\\\\?\\' + str(path))
    return path.is_file()
for column in ['rf_backbone_pdb','boltz_model_0_pdb','boltz_model_1_pdb','boltz_model_2_pdb']:
    assert demo[column].map(package_file_exists).all(), column
print('Formal candidates:', len(demo))
print('Strictly above baseline:', int((demo['score_minus_baseline'] > 0).sum()))
print('Best linker:', demo.iloc[0]['linker_sequence'])
print('Structure mapping: PASS')

In [ ]:
subprocess.run([sys.executable, str(root / 'validate_submission.py')], cwd=root, check=True)
print('Submission validation: PASS')

## 门控逻辑
正式候选同时通过序列QC、linker Cα RMSD三模型至少二个严格小于2.0 Å、Vina三模型至少二个不高于−6.0 kcal/mol，并完成SD40几何评分。所有结果均为计算筛选结果，需要后续实验验证。